# Anti-defense in the dark plasmidome

Does the unannotatable half of the small plasmidome encode **anti-defense** systems — anti-CRISPR,
anti-SOS, anti-restriction?

The hypothesis is a good one. Anti-CRISPR (Acr) proteins are short (50–150 aa), hyperdiverse and
share no common fold — the exact profile of an unannotatable ORF — and plasmids have every reason
to carry them. `psiA`/`psiB` (anti-SOS) and `ardA`/`klcA`/`ocr` (anti-restriction) are classic
conjugative-plasmid cargo.

**Answer: no.** After adjudication the signal is **158 dark ORFs, 0.042% of the dark proteome** —
about three orders of magnitude short of explaining anything. The hypothesis is cleanly rejected
for *known* anti-defense families.

Two things are worth more than the headline:

1. **The raw database output is inflated ~3×** by cross-hits to abundant plasmid housekeeping
   proteins. Reported uncritically, this analysis would have called replication initiators
   anti-CRISPRs.
2. **Classic anti-defense genes are not in the dark fraction at all** — they sit in the *named*
   proteome, where existing tools already find them.

Upstream: `dark_plasmidome.ipynb` (what the dark proteome is), `reports/dark_orf_clustering.md`,
`reports/pfam_dark_validation.md`. This notebook reads finished search outputs; it runs no searches.

## 1. Setup

Two independent searches over the same query set of 92,752 dark family representatives plus 78,397
PlasAnn-named proteins as a positive control:

| search | database | threshold | script |
|---|---|---|---|
| Pfam-A 38.2 | 30,134 families | `--cut_ga` (curated) | `scripts/pfam_array.sbatch` |
| dbAPIS | 290 HMMs (200 APIS + 90 Acr) | none curated — chosen here | `scripts/dbapis_search.sbatch` |

Everything is weighted by **family size**, so a hit on a 1,000-member family is not counted the
same as a hit on a singleton.

In [ ]:
# Anti-defense screen of the dark plasmidome — setup.
# Reads finished search outputs only; nothing here re-runs hmmsearch.
#   Pfam-A 38.2   -> scripts/pfam_array.sbatch      -> data/pfam_run/
#   dbAPIS (290)  -> scripts/dbapis_search.sbatch   -> data/dbapis_run/
import gzip
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "data" / "dbapis_run").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
D, PF, DBR = ROOT / "data/dark_orf_run", ROOT / "data/pfam_run", ROOT / "data/dbapis_run"
PFREF, DBREF = ROOT / "data/refs/pfam", ROOT / "data/refs/dbapis"

S1, S2, S3, S4, NEUT = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#b8b7ae"
INK, INK2, GRID, FAINT = "#0b0b0b", "#52514e", "#dcdbd4", "#d4d3cc"

def tidy(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set(color=GRID, linewidth=1)
    ax.tick_params(colors=INK2, labelsize=9, length=0)
    ax.set_axisbelow(True)

# dark family sizes — every count in this notebook is weighted by family size, so a hit on a
# 1,000-member family is not treated the same as a hit on a singleton.
clus = pd.read_csv(D / "dark30_cluster.tsv", sep="\t", header=None, names=["rep", "member"])
size = clus.groupby("rep").size()
N_DARK = len(clus)

# Pfam best hit per sequence + family metadata.
per = pd.read_csv(PF / "pfam_per_seq.tsv", sep="\t")
per["acc"] = per["pfam_acc"].str.split(".").str[0]
man = dict(l.split("\t") for l in (PF / "query_manifest.txt").read_text().splitlines())
rec, cur = {}, {}
for line in gzip.open(PFREF / "Pfam-A.hmm.dat.gz", "rt"):
    if line.startswith("//"):
        if cur:
            rec[cur.get("AC", "").split(".")[0]] = cur
        cur = {}
    elif line.startswith("#=GF "):
        t, v = line[5:7], line[8:].strip()
        if t in ("ID", "AC", "DE"):
            cur[t] = v
pfmeta = pd.DataFrame(rec).T[["ID", "DE"]]
per = per.join(pfmeta, on="acc")

print(f"dark ORF proteins       {N_DARK:,}  in {size.size:,} families")
print(f"Pfam queries            DARKREP {int(man['DARKREP']):,} | NAMED {int(man['NAMED']):,}")
print(f"dbAPIS domain rows      {sum(1 for l in open(DBR / 'dbapis.domtbl') if not l.startswith('#')):,}")

## 2. Screen 1 — the anti-defense families Pfam already carries

Before reaching for a specialist database, ask what the general one says. Pfam holds a usable
anti-restriction and anti-SOS catalogue (`ArdA`, `ArdcN`, `Antirestrict`, `ocr`, `PsiA`, `PsiB`)
even though its anti-CRISPR coverage is thin (~11 models).

Name collisions are excluded deliberately — `AcrB`/`AcrZ`/`ACR_tran` are efflux pumps and `ABI`/
`Abi_HHR` is the human Abl-interactor. Abortive-infection `Abi*` systems are kept out too: those
are **defense**, not anti-defense.

In [ ]:
# Screen 1 — the known anti-defense families that Pfam itself carries.
# Curated by hand from Pfam-A.clans.tsv. Name collisions are excluded deliberately:
# AcrB / AcrZ / ACR_tran are efflux pumps, ABI / Abi_HHR are the human Abl-interactor, and
# abortive-infection Abi* systems are DEFENSE, not anti-defense.
ANTI = {
 "anti-CRISPR": {"PF20829":"Acr30-35_AcrF1","PF25704":"AcrF2","PF21401":"AcrF3","PF25703":"AcrF31",
   "PF25699":"AcrF8","PF22147":"AcrIC5","PF28100":"AcrIE3-like","PF26151":"AcrIF11_ADPrib",
   "PF24304":"AcrIIA4","PF28149":"AcrIIA6","PF26125":"AcrVA2-like"},
 "anti-restriction": {"PF03230":"Antirestrict","PF07275":"ArdA","PF08401":"ArdcN","PF08684":"ocr",
   "PF11058":"Ral","PF14354":"Lar_restr_allev","PF22134":"DM_Arn"},
 "anti-SOS": {"PF06952":"PsiA","PF06290":"PsiB"},
 "anti-defense (phage systems)": {"PF26840":"DSR_AD1","PF25705":"Gad1","PF26128":"Gad2",
   "PF26126":"Had1","PF11195":"Tad2-like"},
}
acc2cat = {a: c for c, d in ANTI.items() for a in d}
acc2nm = {a: n for c, d in ANTI.items() for a, n in d.items()}
print(f"curated anti-defense Pfam families: {len(acc2cat)} "
      f"({' / '.join(f'{c} {len(d)}' for c, d in ANTI.items())})\n")

dk = per[per["set"].eq("DARKREP")].copy()
dk["rep"] = dk["target"].str.split("::").str[1]
dk["members"] = dk["rep"].map(size)
nm = per[per["set"].eq("NAMED")].copy()
nm["members"] = 1

rows = []
for acc in acc2cat:
    d = dk.loc[dk["acc"].eq(acc), "members"].sum()
    n = int(nm["acc"].eq(acc).sum())
    if d or n:
        rows.append({"family": acc2nm[acc], "category": acc2cat[acc],
                     "dark ORFs": int(d), "named proteins": n})
PFSCREEN = pd.DataFrame(rows).sort_values("named proteins", ascending=False)
print("anti-defense Pfam families actually detected:")
print(PFSCREEN.to_string(index=False))
print(f"\ntotals: {PFSCREEN['dark ORFs'].sum():,} dark ORFs "
      f"({PFSCREEN['dark ORFs'].sum() / N_DARK * 100:.3f}% of the dark proteome) "
      f"vs {PFSCREEN['named proteins'].sum():,} named proteins "
      f"({PFSCREEN['named proteins'].sum() / int(man['NAMED']) * 100:.2f}% of the named proteome)")
print("\nThe classic plasmid anti-defense genes sit in the NAMED proteome, not the dark one.")

In [ ]:
# Figure 1 — where the known anti-defense genes actually live.
fig, ax = plt.subplots(figsize=(10.5, 5.2))
p = PFSCREEN.sort_values("named proteins")
y, h = np.arange(len(p)), 0.36
ax.barh(y + h / 2, p["named proteins"], height=h * 0.9, color=NEUT,
        label=f"named proteome ({int(man['NAMED']):,} proteins)")
ax.barh(y - h / 2, p["dark ORFs"], height=h * 0.9, color=S1,
        label=f"dark proteome ({N_DARK:,} ORFs)")
for i, r in enumerate(p.itertuples()):
    if r._3: ax.text(r._3 + 1.5, i - h / 2, f"{r._3}", va="center", fontsize=8, color=INK2)
    if r._4: ax.text(r._4 + 1.5, i + h / 2, f"{r._4}", va="center", fontsize=8, color=INK2)
ax.set(yticks=y, yticklabels=[f"{r.family}  ({r.category.replace('anti-', 'anti-')})"
                              for r in p.itertuples()],
       xlim=(0, max(p["named proteins"].max(), p["dark ORFs"].max()) * 1.16))
ax.tick_params(axis="y", labelsize=8.5)
ax.set_xlabel("proteins / dark ORFs carrying the domain", fontsize=9, color=INK2)
ax.grid(axis="x", color=GRID, linewidth=1)
ax.legend(frameon=False, fontsize=9, labelcolor=INK2, loc="lower right")
ax.set_title("Screen 1 — known anti-defense families in Pfam\n"
             "the dark proteome is 4.8× larger than the named one, yet carries fewer of these",
             fontsize=11, color=INK, loc="left")
tidy(ax)
plt.tight_layout()
plt.show()

## 3. Screen 2 — dbAPIS, the purpose-built database

**dbAPIS** (anti-Prokaryotic Immune Systems) carries **290 HMMs: 200 APIS families + 90 anti-CRISPR
families** — roughly 8× Pfam's Acr coverage. The GitHub distribution bundles Acr HMMs that are not
even published on the project website.

Two complications, both handled rather than hidden:

1. **No curated gathering thresholds.** Unlike Pfam, dbAPIS models carry no `GA`/`TC`/`NC` lines, so
   a cutoff has to be chosen. Results are reported at three tiers so the choice is visible.
2. **Cross-hits to housekeeping proteins.** Several models fire on abundant plasmid proteins. Every
   hit is adjudicated against its independent Pfam assignment.

The adjudication rule is deliberately aggressive: any hit whose Pfam call is plasmid housekeeping is
discarded. That will also discard genuine anti-defense proteins carrying an HTH or Rep-like domain,
so the surviving number is a **floor**, not an estimate.

In [ ]:
# Screen 2 — dbAPIS, the purpose-built resource: 290 HMMs = 200 APIS families + 90 anti-CRISPR,
# roughly 8x Pfam's Acr coverage. Two caveats drive the whole analysis below:
#   (a) dbAPIS models carry NO curated GA thresholds -> a cutoff must be chosen, so report tiers.
#   (b) several models cross-hit abundant plasmid housekeeping proteins -> adjudicate vs Pfam.
rows = []
for line in open(DBR / "dbapis.domtbl"):
    if line.startswith("#"):
        continue
    p = line.split(None, 22)
    rows.append((p[0], p[3], int(p[5]), float(p[6]), int(p[15]), int(p[16])))
db = pd.DataFrame(rows, columns=["target", "model", "qlen", "evalue", "hmm_from", "hmm_to"])
db["set"] = db["target"].str.split("::").str[0]
db["hmm_cov"] = (db["hmm_to"] - db["hmm_from"] + 1) / db["qlen"]
apis = pd.read_csv(DBREF / "seed_and_familyrep_all_infor.tsv", sep="\t", low_memory=False)
db["inhibits"] = db["model"].map(dict(zip(apis["APIS families"], apis["Defense systems"])))

TIER = [("permissive", 1e-3), ("standard", 1e-5), ("strict", 1e-10)]
trows = []
for name, ev in TIER:
    s = db[(db.evalue <= ev) & (db.hmm_cov >= 0.5) & db.set.eq("DARKREP")].copy()
    s["rep"] = s["target"].str.split("::").str[1]
    s = s.sort_values("evalue").drop_duplicates("rep")
    trows.append({"tier": f"{name}\nE≤{ev:.0e}", "families": len(s),
                  "dark ORFs": int(s["rep"].map(size).sum())})
TIERS = pd.DataFrame(trows).set_index("tier")
TIERS["% of dark"] = TIERS["dark ORFs"] / N_DARK * 100
print("stringency tiers (HMM coverage ≥ 0.5 throughout):")
print(TIERS.round(3).to_string())
print("  -> the answer does not depend on the cutoff.\n")

dbk = db[(db.evalue <= 1e-5) & (db.hmm_cov >= 0.5) & db.set.eq("DARKREP")].copy()
dbk["rep"] = dbk["target"].str.split("::").str[1]
dbk = dbk.sort_values("evalue").drop_duplicates("rep")
dbk["members"] = dbk["rep"].map(size)
dbk = dbk.merge(dk[["rep", "ID"]], on="rep", how="left").rename(columns={"ID": "pfamID"})

CONFOUND = re.compile(r"^(RepA_N|Bac_RepA_C|Rep_|Rep3|RepL|Replicase|SSB|MethyltransfD12|"
                      r"DNA_methylase|N6_M|Mob_|MobA|Relaxase|NikA|HTH_|Resolvase|"
                      r"Phage_integrase|MPTase)", re.I)
ANTI_OK = re.compile(r"^(ArdA|ArdcN|Antirestrict|Psi[AB]|ocr|Ral|Lar_|DM_Arn|Acr)", re.I)

def adjudicate(r):
    if pd.isna(r.pfamID):         return "no Pfam hit (unadjudicated)"
    if ANTI_OK.search(r.pfamID):  return "Pfam agrees (anti-defense)"
    if CONFOUND.search(r.pfamID): return "Pfam contradicts (housekeeping)"
    return "Pfam names something else"
dbk["verdict"] = dbk.apply(adjudicate, axis=1)

ADJ = (dbk.groupby("verdict").agg(families=("rep", "size"), dark_ORFs=("members", "sum"))
       .sort_values("dark_ORFs", ascending=False))
ADJ["% of hits"] = ADJ["dark_ORFs"] / ADJ["dark_ORFs"].sum() * 100
print(f"adjudicating the {len(dbk)} dark hits at the standard tier against Pfam:")
print(ADJ.round(1).to_string())

FP = (dbk[dbk.verdict.str.startswith("Pfam contradicts")]
      .groupby(["model", "pfamID"])["members"].sum().sort_values(ascending=False))
print("\nthe false positives — dbAPIS model vs what Pfam actually calls the protein:")
print(FP.to_string())

SURV = dbk[~dbk.verdict.str.startswith("Pfam contradicts")].copy()
SURV["system"] = np.where(SURV["model"].str.startswith("Acr"), "CRISPR-Cas (anti-CRISPR)",
                          SURV["inhibits"].fillna("(system not recorded)"))
BYSYS = SURV.groupby("system")["members"].sum().sort_values(ascending=False)
ACR = SURV[SURV["model"].str.startswith("Acr")].groupby("model")["members"].sum() \
        .sort_values(ascending=False)
print(f"\nSURVIVING: {len(SURV)} families, {SURV['members'].sum():,} dark ORFs = "
      f"{SURV['members'].sum() / N_DARK * 100:.3f}% of the dark proteome")
print(BYSYS.to_string())
print(f"\nanti-CRISPR only: {len(ACR)} models, {ACR.sum()} dark ORFs "
      f"({ACR.sum() / N_DARK * 100:.3f}%)")
print(ACR.to_string())

In [ ]:
# Figure 2 — the cutoff does not matter (A), but the adjudication does (B).
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1, 1.35]})

x = np.arange(len(TIERS))
axA.bar(x, TIERS["dark ORFs"], width=0.45, color=S1)
for i, (v, f) in enumerate(zip(TIERS["dark ORFs"], TIERS["families"])):
    axA.text(i, v + TIERS["dark ORFs"].max() * 0.03,
             f"{v}\n{f} fam\n{v / N_DARK * 100:.3f}%", ha="center", fontsize=8.5, color=INK2)
axA.set(xticks=x, xticklabels=TIERS.index, yticks=[], ylim=(0, TIERS["dark ORFs"].max() * 1.42))
axA.set_xlabel("dbAPIS E-value threshold", fontsize=9, color=INK2)
axA.set_title("A · Raw dbAPIS hits by stringency\ndbAPIS has no curated thresholds, so the "
              "cutoff is ours —\nthe answer barely moves across three tiers",
              fontsize=10.5, color=INK, loc="left")
tidy(axA)

order = ["Pfam contradicts (housekeeping)", "no Pfam hit (unadjudicated)",
         "Pfam names something else", "Pfam agrees (anti-defense)"]
cols = {order[0]: NEUT, order[1]: FAINT, order[2]: S4, order[3]: S1}
left = 0.0
for k in order:
    v = ADJ.loc[k, "dark_ORFs"]
    axB.barh([0], [v], left=left, height=0.40, color=cols[k], edgecolor="white", linewidth=1.5,
             label=f"{k} — {v}")
    if v > 40:
        axB.text(left + v / 2, 0, f"{v}", ha="center", va="center", fontsize=9,
                 color="white" if k in (order[0], order[3]) else INK)
    left += v
axB.set(xlim=(0, left), yticks=[], ylim=(-0.36, 0.36))
axB.set_xlabel(f"dark ORFs among the {len(dbk)} standard-tier hits", fontsize=9, color=INK2)
axB.legend(frameon=False, fontsize=8, labelcolor=INK2, ncol=2,
           loc="upper center", bbox_to_anchor=(0.5, -0.24))
axB.set_title(f"B · Every hit adjudicated against Pfam\n"
              f"{ADJ.loc[order[0], '% of hits']:.0f}% of the raw signal is a cross-hit to a "
              f"plasmid housekeeping protein", fontsize=10.5, color=INK, loc="left")
tidy(axB)

plt.tight_layout()
plt.show()

In [ ]:
# Figure 3 — the anatomy of the false positives. This is the figure that stops someone
# repeating the analysis and reporting 3x the real number.
fig, ax = plt.subplots(figsize=(11, 4.4))
f = FP.iloc[::-1]
labels = [f"{m}  →  {p}" for m, p in f.index]
y = np.arange(len(f))
ax.barh(y, f.values, height=0.44, color=NEUT)
for i, v in enumerate(f.values):
    ax.text(v + f.max() * 0.015, i, f"{v}", va="center", fontsize=9, color=INK2)
ax.set(yticks=y, yticklabels=labels, xlim=(0, f.max() * 1.14))
ax.tick_params(axis="y", labelsize=9)
ax.set_xlabel("dark ORFs wrongly called anti-defense", fontsize=9, color=INK2)
ax.grid(axis="x", color=GRID, linewidth=1)
ax.set_title("Figure 3 · What the discarded hits really are\n"
             "left = the dbAPIS model that fired · right = what Pfam calls the same protein\n"
             "the largest single \"anti-CRISPR\" signal, AcrIIA21, is a replication initiator",
             fontsize=10.5, color=INK, loc="left")
tidy(ax)
plt.tight_layout()
plt.show()

## 4. What survives

In [ ]:
# Figure 4 — what actually survives adjudication, by inhibited system (A) and by Acr model (B).
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1.3, 1]})

b = BYSYS.iloc[::-1]
y = np.arange(len(b))
axA.barh(y, b.values, height=0.44,
         color=[FAINT if k.startswith("(system") else S1 for k in b.index])
for i, v in enumerate(b.values):
    axA.text(v + b.max() * 0.02, i, f"{v}", va="center", fontsize=9, color=INK2)
axA.set(yticks=y, yticklabels=b.index, xlim=(0, b.max() * 1.18))
axA.tick_params(axis="y", labelsize=9)
axA.set_xlabel("dark ORFs", fontsize=9, color=INK2)
axA.set_title(f"A · Surviving hits by defense system inhibited\n"
              f"{SURV['members'].sum():,} dark ORFs total — "
              f"{SURV['members'].sum() / N_DARK * 100:.3f}% of the dark proteome",
              fontsize=10.5, color=INK, loc="left")
tidy(axA)

a = ACR.iloc[::-1]
ya = np.arange(len(a))
axB.barh(ya, a.values, height=0.44, color=S1)
for i, v in enumerate(a.values):
    axB.text(v + a.max() * 0.02, i, f"{v}", va="center", fontsize=9, color=INK2)
axB.set(yticks=ya, yticklabels=a.index, xlim=(0, a.max() * 1.2))
axB.set_xlabel("dark ORFs", fontsize=9, color=INK2)
axB.set_title(f"B · Anti-CRISPR models only\n{len(ACR)} of the 90 Acr models fire at all; "
              f"{ACR.sum()} ORFs ({ACR.sum() / N_DARK * 100:.3f}%)",
              fontsize=10.5, color=INK, loc="left")
tidy(axB)

plt.tight_layout()
plt.show()

## 5. The result in proportion

In [ ]:
# Figure 5 — the result in proportion. Log scale, because the whole point is that the
# anti-defense signal is three orders of magnitude below the thing it was meant to explain.
PFHIT = int(dk.loc[dk["acc"].notna(), "members"].sum())
steps = [("dark proteome", N_DARK, NEUT),
         ("…Pfam can name", PFHIT, FAINT),
         ("raw dbAPIS hits", int(TIERS.loc[TIERS.index[1], "dark ORFs"]), S4),
         ("…surviving adjudication", int(SURV["members"].sum()), S1),
         ("…anti-CRISPR only", int(ACR.sum()), S2)]

fig, ax = plt.subplots(figsize=(11, 3.9))
y = np.arange(len(steps))[::-1]
ax.barh(y, [v for _, v, _ in steps], height=0.5, color=[c for _, _, c in steps])
for yi, (lab, v, _) in zip(y, steps):
    ax.text(v * 1.15, yi, f"{v:,}   ({v / N_DARK * 100:.3f}% of dark)",
            va="center", fontsize=9, color=INK2)
ax.set(yticks=y, yticklabels=[s[0] for s in steps], xscale="log", xlim=(30, N_DARK * 12))
ax.tick_params(axis="y", labelsize=9.5)
ax.set_xlabel("dark ORFs (log scale)", fontsize=9, color=INK2)
ax.grid(axis="x", color=GRID, linewidth=1)
ax.set_title("Figure 5 · The anti-defense hypothesis, in proportion\n"
             "log scale — anti-defense is ~3 orders of magnitude short of explaining "
             "the dark proteome", fontsize=11, color=INK, loc="left")
tidy(ax)
plt.tight_layout()
plt.show()

## 6. Screen 3 — DefenseFinder + AntiDefenseFinder

A third, fully independent test: **DefenseFinder 3.0.0** (Tesson et al.) with its
`--antidefensefinder` models. Different software, different model set, and — crucially — a
different *evidence rule*: MacSyFinder uses **genomic context**, calling a system only when its
component genes co-localise. dbAPIS and Pfam are pure homology.

That required running on **whole plasmids in gene order**, not on the clustered family
representatives used above. Input is MacSyFinder `gembase` format built by
`scripts/build_defensefinder_input.py`; plasmids are renamed `PL######` because macsypy derives
the replicon name by stripping everything after the last underscore and then `.rstrip('ib')`,
which mangles real accessions.

65,805 plasmids / 456,949 proteins, 33-task SLURM array, ~2 minutes.

This screen also answers a question the payload filter never asked. "Payload-free" was defined as
no AMR, no virulence, no metal/biocide — **defense systems were never in that list.**

In [ ]:
# Screen 3 — DefenseFinder 3.0.0 + AntiDefenseFinder, run on WHOLE PLASMIDS in gene order
# (MacSyFinder gembase), so genomic context is available: a system is only called when its
# component genes co-localise. Search: scripts/defensefinder_array.sbatch (33-task array).
# This is an INDEPENDENT second opinion on the dbAPIS result above — different models,
# different software, different evidence rule.
from collections import defaultdict
DFR = ROOT / "data" / "defensefinder_run"

sysd = pd.read_csv(DFR / "df_systems.tsv", sep="\t")
gns = pd.read_csv(DFR / "df_genes.tsv", sep="\t")
N_SEARCHED = 65_805          # payload-free small plasmids that have PlasAnn genes

# Map each gembase gene id back to the original plasmid + CDS + dark/named status.
by = defaultdict(list)
for line in open(D / "all_cds.faa"):
    if line.startswith(">"):
        pid, idx, kind = line[1:].strip().rsplit("|", 2)
        by[pid].append((int(idx), kind))
p2g = dict(zip(*pd.read_csv(DFR / "plasmid_id_map.tsv", sep="\t")
                .pipe(lambda d: (d["plasmid_id"], d["gembase_id"]))))
lookup = {}
for pid, lst in by.items():
    for i, (idx, kind) in enumerate(sorted(lst), 1):
        lookup[f"{p2g[pid]}_{i:05d}"] = kind
gns["activity"] = gns["sys_id"].map(dict(zip(sysd["sys_id"], sysd["activity"])))
gns["kind"] = gns["hit_id"].map(lookup)

print(f"systems called: {len(sysd):,} on {sysd['plasmid_id'].nunique():,} plasmids "
      f"({sysd['plasmid_id'].nunique() / N_SEARCHED * 100:.2f}% of {N_SEARCHED:,} searched)\n")
print(sysd["activity"].value_counts().to_string())

DEF = sysd[sysd.activity.eq("Defense")]
DEFTYPE = (DEF.groupby("type").agg(systems=("sys_id", "size"), plasmids=("plasmid_id", "nunique"),
                                   multi=("genes_count", lambda x: (x > 1).sum()))
           .sort_values("systems", ascending=False))
DEFTYPE["multi-gene"] = DEFTYPE["multi"] / DEFTYPE["systems"] * 100
print(f"\n=== DEFENSE: {len(DEF):,} systems on {DEF['plasmid_id'].nunique():,} plasmids "
      f"({DEF['plasmid_id'].nunique() / N_SEARCHED * 100:.1f}%) ===")
print(DEFTYPE.head(14).round(0).to_string())
print(f"\n{(DEF.genes_count == 1).mean() * 100:.1f}% of Defense calls are single-gene — but for "
      f"RM_Type_IIG, AbiQ, HEC-09 and PD-T4-5 that is the correct architecture,\nnot weak evidence. "
      f"Multi-gene calls: {(DEF.genes_count > 1).sum():,} on "
      f"{DEF[DEF.genes_count > 1]['plasmid_id'].nunique():,} plasmids.")

# --- AntiDefenseFinder, and the same false positives as dbAPIS -------------------
ADF = sysd[sysd.activity.eq("Antidefense")]
ADSUB = (ADF.groupby(["type", "subtype"]).agg(systems=("sys_id", "size"),
                                              plasmids=("plasmid_id", "nunique"))
         .sort_values("systems", ascending=False))
print(f"\n=== ANTIDEFENSE: {len(ADF):,} systems on {ADF['plasmid_id'].nunique():,} plasmids ===")
print(f"{(ADF.genes_count == 1).mean() * 100:.0f}% are single-gene — genomic context adds "
      f"essentially nothing here, so this is homology only.")
print(ADSUB.head(12).to_string())

FP_GENES = gns["gene_name"].str.contains("acriia21|ardc", case=False, na=False)
print(f"\nthe two dominant antidefense calls are the SAME false positives dbAPIS produced:")
print(f"  acriia21 : {int(ADSUB.loc[('Anti_CRISPR', 'acriia21'), 'systems'])} systems — "
      f"165/167 gene calls are Pfam RepA_N / Bac_RepA_C (replication initiator)")
print(f"  ardc     : {int(ADSUB.loc[('Anti_RM', 'ardc'), 'systems'])} systems — "
      f"64/67 gene calls are Pfam MPTase-PolyVal (metallopeptidase)")

dk_g = gns[gns.kind.eq("D")]
dk_ok = gns[gns.kind.eq("D") & ~FP_GENES]
DARKCONTRIB = pd.Series({
    "Defense": int((dk_ok.activity == "Defense").sum()),
    "Antidefense (FPs removed)": int((dk_ok.activity == "Antidefense").sum()),
})
print(f"\n=== contribution to the dark proteome ({N_DARK:,} ORFs) ===")
print(f"dark ORFs in a DefenseFinder system: {len(dk_g):,} ({len(dk_g) / N_DARK * 100:.3f}%)")
for k, v in DARKCONTRIB.items():
    print(f"  {k:<28} {v:>6,}  ({v / N_DARK * 100:.3f}%)")
ratio = DARKCONTRIB["Defense"] / max(DARKCONTRIB["Antidefense (FPs removed)"], 1)
print(f"\nDefense explains ~{ratio:.0f}x more of the dark proteome than antidefense does.")

In [ ]:
# Figure 6 — what DefenseFinder found. Colour marks the EVIDENCE, not the category:
# blue = system called from co-located genes; grey = single-gene call. Panel B uses
# orange for verified false positives so grey keeps ONE meaning across both panels.
# For RM_Type_IIG / AbiQ / HEC-09 / PD-T4-5 a single gene IS the system, so grey here means
# "homology only", not "wrong".
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13.5, 5.4), gridspec_kw={"width_ratios": [1.3, 1]})

t = DEFTYPE.head(14).iloc[::-1]
y = np.arange(len(t))
axA.barh(y, t["multi"], height=0.42, color=S1, label="multi-gene (co-located)")
axA.barh(y, t["systems"] - t["multi"], left=t["multi"], height=0.42, color=NEUT,
         label="single-gene call")
for i, v in enumerate(t["systems"]):
    axA.text(v + t["systems"].max() * 0.015, i, f"{v:,}", va="center", fontsize=8, color=INK2)
axA.set(yticks=y, yticklabels=t.index, xlim=(0, t["systems"].max() * 1.14))
axA.tick_params(axis="y", labelsize=9)
axA.set_xlabel("systems called", fontsize=9, color=INK2)
axA.legend(frameon=False, fontsize=8.5, labelcolor=INK2, loc="lower right")
axA.set_title(f"A · Defense systems on payload-free plasmids\n"
              f"{len(DEF):,} systems on {DEF['plasmid_id'].nunique():,} plasmids "
              f"({DEF['plasmid_id'].nunique() / N_SEARCHED * 100:.1f}% of those searched) — "
              f"restriction-modification dominates", fontsize=10.5, color=INK, loc="left")
tidy(axA)

a = ADSUB.head(10).reset_index()
a["fp"] = a["subtype"].isin(["acriia21", "ardc"])
a = a.iloc[::-1]
ya = np.arange(len(a))
axB.barh(ya, a["systems"], height=0.42, color=[S2 if f else S1 for f in a["fp"]])
for i, (v, f) in enumerate(zip(a["systems"], a["fp"])):
    axB.text(v + a["systems"].max() * 0.02, i, f"{v}" + ("  ✗" if f else ""),
             va="center", fontsize=8, color=INK2)
axB.set(yticks=ya, yticklabels=[f"{r.subtype}  ({r.type.replace('Anti_', 'anti-')})"
                                for r in a.itertuples()],
        xlim=(0, a["systems"].max() * 1.2))
axB.tick_params(axis="y", labelsize=8.5)
axB.set_xlabel("systems called", fontsize=9, color=INK2)
axB.set_title("B · AntiDefenseFinder — orange ✗ = verified false positive\n"
              "the two largest calls are RepA replication initiators and a\n"
              "metallopeptidase, exactly as dbAPIS produced",
              fontsize=10.5, color=INK, loc="left")
tidy(axB)

plt.tight_layout()
plt.show()

### DefenseFinder verdict

**The anti-defense negative holds, from a third independent direction.** AntiDefenseFinder calls
435 systems, but **94% are single-gene**, so genomic context contributes essentially nothing and
this reduces to homology. More tellingly, its two largest calls are the *same false positives*
dbAPIS produced — `acriia21` (165/167 gene calls are Pfam `RepA_N`/`Bac_RepA_C` replication
initiators) and `ardc` (64/67 are a metallopeptidase). Two independent tools, the same conclusion
and the same artifacts. After removing them, antidefense accounts for **91 dark ORFs, 0.024%** of
the dark proteome — consistent with the 158 from dbAPIS.

**The unexpected result is on the defense side.** These "payload-free" plasmids carry
**3,647 defense systems on 3,362 plasmids — 5.1% of those searched**, dominated by
restriction-modification (`RM_Type_II` 1,074, `RM_Type_IIG` 438) plus `MazEF`, `HEC-09`, `AbiQ`,
`Thoeris`, `Kiwa`, `CBASS` and `Retron`. That accounts for **5,008 dark ORFs (1.32% of the dark
proteome) — about 55× more than anti-defense.**

This is a genuine gap in the payload definition rather than a contradiction of it: `card_n_arg`,
`plasann_n_virulence` and `plasann_n_metal_biocide` do not look for defense systems, so a plasmid
carrying a complete Type II RM system is still "payload-free" by that definition. Worth a sentence
in any write-up that uses the term.

**Caveats.** 43.9% of Defense calls are single-gene — for `RM_Type_IIG`, `AbiQ`, `HEC-09` and
`PD-T4-5` that is the correct architecture, not weak evidence, but it does mean those rest on
homology alone; the context-supported subset is 2,045 systems on 1,956 plasmids (2.97%). And
defense systems still explain only ~1.3% of the dark proteome — like anti-defense, they are not
the answer to what the dark plasmidome is, just a larger slice of it than the hypothesis under
test.

## Conclusion, and what this does not show

**Anti-defense does not explain the dark plasmidome.** 158 dark ORFs (0.042%); anti-CRISPR alone is
63 (0.017%). Stable across every stringency tier.

**Anti-SOS is not hidden — it is annotated.** `psiA`/`psiB` appear 69 times in the named proteome
against ~15 dark ORFs' worth. Where plasmids carry classic anti-defense genes, current tools find
them; they are not part of the dark fraction. The same holds for `ardA`/`ardC`/`Antirestrict`.

### Limits

- **This rules out *known* families, not the hypothesis.** dbAPIS holds 90 Acr families; the true
  number is unknown and larger. A novel Acr with no database homolog is invisible to every search
  here — and that is precisely the population the dark proteome would contain if the hypothesis
  were right. Absence of evidence, bounded by database coverage.
- **The adjudication regex is ours** and deliberately aggressive; 158 is a floor.
- **No genomic context was used.** Acr genes typically sit beside an HTH-containing `aca` gene, and
  the PlasAnn shards carry CDS coordinates — a guilt-by-association scan is feasible and would be
  independent of homology. It is the one test that could support the hypothesis where sequence
  search structurally cannot. Not run here.
- **Structure beats sequence for short hyperdiverse proteins.** `foldseek` against the AlphaFold DB
  would serve both this question and the novelty ceiling in `dark_plasmidome.ipynb` §4.